<a href="https://colab.research.google.com/github/hillarymaximino/metodos_numericos/blob/main/Atividade_12_M%C3%A9todos_Num%C3%A9ricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import scipy.linalg as sla

np.set_printoptions(precision=8, suppress=True)


### Exercício de Fixação 2

Explique o que acontece com o sistema do Exemplo 2 quando $\varepsilon \to \infty$.

O sistema é:

$$
\begin{bmatrix}
\varepsilon & 2 \\
1 & 2
\end{bmatrix}
\begin{bmatrix}
x_1 \\ x_2
\end{bmatrix}
=
\begin{bmatrix}
4 \\ 3
\end{bmatrix}
$$

Temos que:

$$
x_2 = \frac{3\varepsilon - 4}{2\varepsilon - 2}
$$

Quando $\varepsilon$ tende ao infinito, dividindo os termos por $\varepsilon$, encontramos $x_2 \to \frac{3}{2}$.

Como $x_1 = \frac{4 - 2x_2}{\varepsilon}$, o valor de $x_1$ tende a zero.

Portanto, quando $\varepsilon \to \infty$, a solução tende a $(0; 1,5)$.

## Exercício de Fixação 1

Explique com suas palavras em que situações a eliminação gaussiana convencional pode gerar problemas computacionais, descrevendo, inclusive, que problemas podem aparecer.

A eliminação gaussiana convencional pode apresentar problemas quando o pivô escolhido é igual a zero ou tem valor muito pequeno em comparação com os demais elementos da coluna. Se o pivô for zero, ocorre uma divisão por zero e o algoritmo não consegue continuar. Se for muito pequeno, os multiplicadores podem ficar muito grandes e aumentar os erros de arredondamento, causando perda de precisão nos resultados. O pivotamento parcial ajuda a evitar esses problemas, pois escolhe o elemento de maior valor absoluto na coluna, a partir da linha do pivô.

## Exercício de Fixação 3

Explique por que não podemos simplesmente trocar os elementos relevantes das linhas \(k\) e \(p\) usando o código abaixo. Descreva o que acontece e explique por que a solução apresentada no texto funciona.

```python
for j in range(k, n):
    A[k, j] = A[p, j]
    A[p, j] = A[k, j]
```

O mesmo problema ocorre com o fatiamento:

```python
A[k, k:] = A[p, k:]
A[p, k:] = A[k, k:]
```

Na primeira atribuição, os valores da linha `k` são substituídos pelos valores da linha `p`. Na segunda, a linha `p` recebe esses mesmos valores, e os valores originais da linha `k` são perdidos. Para trocar as linhas corretamente, podemos usar:

```python
A[[k, p]] = A[[p, k]]
```

Essa atribuição realiza a troca usando os valores originais das duas linhas.

## Exercício de Revisão 4

Em que situações é necessário o uso de pivotamento na eliminação gaussiana?

O pivotamento é necessário quando o pivô atual é zero, pois não é possível dividir por zero. Também é recomendado quando o pivô tem valor absoluto muito pequeno em comparação com outros elementos da coluna. A troca de linhas pode evitar divisões problemáticas e diminuir a influência dos erros de arredondamento.

## Exercício de Revisão 5

Explique qual a diferença entre a implementação de pivotamento parcial e completo.

No pivotamento **parcial**, procura-se o maior valor absoluto na coluna atual, considerando a linha do pivô e as linhas abaixo. Se necessário, trocam-se as linhas. No pivotamento **completo**, procura-se o maior valor absoluto em toda a parte ainda não escalonada da matriz, podendo trocar linhas e colunas. Como a troca de colunas altera a ordem das incógnitas, é preciso controlar essa permutação para organizar corretamente o vetor solução.

## Exercício de Revisão 6

Descreva como é feito o processo de eliminação gaussiana com pivotamento parcial.

1. Na coluna atual `k`, procura-se o elemento de maior valor absoluto entre a linha `k` e a última linha.
2. Identifica-se a linha `p` onde esse elemento está.
3. Se `p` for diferente de `k`, trocam-se as linhas `p` e `k`.
4. Usa-se o elemento da posição `[k, k]` como pivô para zerar os elementos abaixo dele.
5. Repete-se o processo nas próximas colunas.
6. Ao obter uma matriz triangular superior, resolve-se o sistema por substituição retroativa.

## Problema 7

Reescreva o algoritmo de eliminação gaussiana para realizar o pivotamento e implemente uma nova função `escalona` com essa possibilidade, usando `argmax` e `max`.


In [ ]:
def escalona_com_pivotamento(matriz, termos_independentes, tolerancia=1e-12):
    A = np.array(matriz, dtype=float, copy=True)
    b = np.array(termos_independentes, dtype=float, copy=True)
    n = len(b)

    if A.shape != (n, n):
        raise ValueError("A matriz deve ser quadrada e compatível com o vetor b.")

    for k in range(n - 1):

        indice_relativo = np.abs(A[k:, k]).argmax()
        pivo_linha = indice_relativo + k
        maior_valor = np.abs(A[k:, k]).max()

        if maior_valor < tolerancia:
            raise ValueError("Sistema singular ou pivô numericamente nulo.")

        if pivo_linha != k:
            A[[k, pivo_linha]] = A[[pivo_linha, k]]
            b[[k, pivo_linha]] = b[[pivo_linha, k]]

        for linha in range(k + 1, n):
            multiplicador = A[linha, k] / A[k, k]
            A[linha, k:] = A[linha, k:] - multiplicador * A[k, k:]
            b[linha] = b[linha] - multiplicador * b[k]
            A[linha, k] = 0.0

    if abs(A[n - 1, n - 1]) < tolerancia:
        raise ValueError("Sistema singular ou último pivô numericamente nulo.")

    return A, b

def substituicao_retroativa(matriz_triangular, termos_transformados):
    A = np.asarray(matriz_triangular, dtype=float)
    b = np.asarray(termos_transformados, dtype=float)
    n = len(b)
    solucao = np.zeros(n)

    for i in range(n - 1, -1, -1):
        soma = np.dot(A[i, i + 1:], solucao[i + 1:])
        if abs(A[i, i]) < 1e-12:
            raise ValueError("Pivô nulo durante a substituição retroativa.")
        solucao[i] = (b[i] - soma) / A[i, i]

    return solucao

matriz_teste = np.array([[1, 1, 1], [4, 4, 2], [2, 1, -1]], dtype=float)
vetor_teste = np.array([1, 2, 0], dtype=float)

matriz_escalonada, vetor_escalonado = escalona_com_pivotamento(matriz_teste, vetor_teste)
solucao_teste = substituicao_retroativa(matriz_escalonada, vetor_escalonado)

print("Matriz escalonada:")
print(matriz_escalonada)
print("Vetor escalonado:", vetor_escalonado)
print("Solução:", solucao_teste)
print("Conferência A @ x:", matriz_teste @ solucao_teste)


## Problema 8

Adapte a implementação do problema anterior para utilizar a função `numpy.roll()` na operação de permutação das linhas, nos casos necessários.


In [ ]:
def escalona_com_roll(matriz, termos_independentes, tolerancia=1e-12):
    A = np.array(matriz, dtype=float, copy=True)
    b = np.array(termos_independentes, dtype=float, copy=True)
    n = len(b)

    if A.shape != (n, n):
        raise ValueError("A matriz deve ser quadrada e compatível com o vetor b.")

    for k in range(n - 1):
        pivo_linha = np.abs(A[k:, k]).argmax() + k

        if abs(A[pivo_linha, k]) < tolerancia:
            raise ValueError("Sistema singular ou pivô numericamente nulo.")

        if pivo_linha != k:

            A[k:pivo_linha + 1] = np.roll(A[k:pivo_linha + 1], 1, axis=0)
            b[k:pivo_linha + 1] = np.roll(b[k:pivo_linha + 1], 1)

        for linha in range(k + 1, n):
            multiplicador = A[linha, k] / A[k, k]
            A[linha, k:] -= multiplicador * A[k, k:]
            b[linha] -= multiplicador * b[k]
            A[linha, k] = 0.0

    if abs(A[-1, -1]) < tolerancia:
        raise ValueError("Sistema singular ou último pivô numericamente nulo.")

    return A, b

matriz_roll, vetor_roll = escalona_com_roll(matriz_teste, vetor_teste)
solucao_roll = substituicao_retroativa(matriz_roll, vetor_roll)
print("Matriz escalonada com roll:")
print(matriz_roll)
print("Vetor escalonado:", vetor_roll)
print("Solução:", solucao_roll)
print("Conferência A @ x:", matriz_teste @ solucao_roll)


### Problema 9

Resolva o sistema $Ax=b$, sendo:

$$
A =
\begin{bmatrix}
4 & 2 & 1 \\
9 & 3 & 1 \\
25 & 5 & 1
\end{bmatrix}
\qquad
b =
\begin{bmatrix}
0.693147 \\
1.098612 \\
1.609438
\end{bmatrix}
$$

Resolva primeiro pelo método da eliminação gaussiana convencional e depois usando o pivotamento parcial.

Durante os cálculos, arredonde os valores intermediários para quatro algarismos significativos, sem utilizar frações.

Depois, calcule o resíduo

$$
r = b - Ax
$$

e sua norma infinito

$$
\|r\|_\infty
$$

Esses dois últimos cálculos devem ser feitos sem arredondamento.

In [ ]:
def arredondar_significativos(valor, algarismos=4):
    valor = float(valor)
    if valor == 0 or not np.isfinite(valor):
        return valor
    casas = algarismos - 1 - int(np.floor(np.log10(abs(valor))))
    return float(np.round(valor, casas))

def operar_arredondando(a, b, operacao):
    if operacao == "soma":
        return arredondar_significativos(a + b)
    if operacao == "subtracao":
        return arredondar_significativos(a - b)
    if operacao == "multiplicacao":
        return arredondar_significativos(a * b)
    if operacao == "divisao":
        return arredondar_significativos(a / b)
    raise ValueError("Operação desconhecida.")

def eliminacao_com_arredondamento(matriz, termos, usar_pivotamento=False):
    A = np.array(matriz, dtype=float, copy=True)
    b = np.array(termos, dtype=float, copy=True)
    n = len(b)

    A = np.vectorize(arredondar_significativos)(A)
    b = np.vectorize(arredondar_significativos)(b)

    for k in range(n - 1):
        if usar_pivotamento:
            p = np.abs(A[k:, k]).argmax() + k
            if p != k:
                A[[k, p]] = A[[p, k]]
                b[[k, p]] = b[[p, k]]

        if A[k, k] == 0:
            raise ZeroDivisionError("Pivô zero na eliminação convencional.")

        for i in range(k + 1, n):
            multiplicador = operar_arredondando(A[i, k], A[k, k], "divisao")
            for j in range(k, n):
                produto = operar_arredondando(multiplicador, A[k, j], "multiplicacao")
                A[i, j] = operar_arredondando(A[i, j], produto, "subtracao")
            produto_b = operar_arredondando(multiplicador, b[k], "multiplicacao")
            b[i] = operar_arredondando(b[i], produto_b, "subtracao")
            A[i, k] = 0.0

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = 0.0
        for j in range(i + 1, n):
            produto = operar_arredondando(A[i, j], x[j], "multiplicacao")
            soma = operar_arredondando(soma, produto, "soma")
        numerador = operar_arredondando(b[i], soma, "subtracao")
        x[i] = operar_arredondando(numerador, A[i, i], "divisao")

    return A, b, x

matriz = np.array([[4, 2, 1],
                   [9, 3, 1],
                   [25, 5, 1]], dtype=float)
termos = np.array([0.693147, 1.098612, 1.609438], dtype=float)

try:
    matriz_convencional, termos_convencionais, solucao_convencional = eliminacao_com_arredondamento(
        matriz, termos, usar_pivotamento=False
    )
    residuo_convencional = termos - matriz @ solucao_convencional
    norma_convencional = np.linalg.norm(residuo_convencional, ord=np.inf)

    print("=== Eliminação convencional ===")
    print("Matriz escalonada:")
    print(matriz_convencional)
    print("Termos transformados:", termos_convencionais)
    print("Solução aproximada:", solucao_convencional)
    print("Resíduo calculado sem arredondamento:", residuo_convencional)
    print("Norma infinito do resíduo:", format(norma_convencional, ".16e"))
except (ZeroDivisionError, ValueError) as erro:
    print("Falha na eliminação convencional:", erro)

matriz_pivotada, termos_pivotados, solucao_pivotada = eliminacao_com_arredondamento(
    matriz, termos, usar_pivotamento=True
)
residuo_pivotado = termos - matriz @ solucao_pivotada
norma_pivotada = np.linalg.norm(residuo_pivotado, ord=np.inf)

print("\n=== Eliminação com pivotamento parcial ===")
print("Matriz escalonada:")
print(matriz_pivotada)
print("Termos transformados:", termos_pivotados)
print("Solução aproximada:", solucao_pivotada)
print("Resíduo calculado sem arredondamento:", residuo_pivotado)
print("Norma infinito do resíduo:", format(norma_pivotada, ".16e"))

solucao_referencia = np.linalg.solve(matriz, termos)
print("\nSolução de referência (NumPy, sem arredondamento manual):", solucao_referencia)
